In [ ]:
import os
import re
import json
import time
import base64
import mimetypes
import pandas as pd
from tqdm import tqdm
from anthropic import Anthropic

import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================

IMAGE_DIR = r"google_test_images/SMER_ViT"
INPUT_CSV = r"data/Labels_Top5.csv"
OUTPUT_CSV = r"data/Labels_Top5_Claude_Adv_SMER.csv"

MODEL = "claude-sonnet-4-6"

MAX_RETRIES = 5
SLEEP_BETWEEN_REQUESTS = 0.3


# ============================================================
# CLIENT
# ============================================================

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
client = Anthropic(
    api_key=ANTHROPIC_API_KEY
)



# ============================================================
# PROMPT
# ============================================================

PROMPT = """
Examine this image carefully.

Identify the main object shown in the image.

Give exactly five most likely object categories, ordered from
most likely to least likely.

Rules:
- Give exactly 5 guesses.
- Use short common object/category names.
- Do not explain your reasoning.
- Do not use numbering.
- Return valid JSON only.

Use exactly this format:

{
  "top5": [
    "label1",
    "label2",
    "label3",
    "label4",
    "label5"
  ]
}
"""


# ============================================================
# IMAGE TO BASE64
# ============================================================

def encode_image(image_path):

    mime_type, _ = mimetypes.guess_type(image_path)

    if mime_type is None:
        mime_type = "image/png"

    # Claude supports common image MIME types
    if mime_type == "image/jpg":
        mime_type = "image/jpeg"

    with open(image_path, "rb") as f:
        encoded = base64.standard_b64encode(
            f.read()
        ).decode("utf-8")

    return encoded, mime_type


# ============================================================
# PARSE TOP-5
# ============================================================

def parse_top5(text):

    if text is None:
        return None

    text = text.strip()

    text = re.sub(
        r"^```json\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"^```\s*",
        "",
        text
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    # First try JSON
    try:

        data = json.loads(text)

        labels = data.get("top5", [])

        if isinstance(labels, list) and len(labels) >= 5:

            return [
                str(x).strip()
                for x in labels[:5]
            ]

    except Exception:
        pass


    # Fallback
    quoted = re.findall(
        r'"([^"]+)"',
        text
    )

    quoted = [
        x for x in quoted
        if x.lower() != "top5"
    ]

    if len(quoted) >= 5:
        return quoted[:5]

    return None


# ============================================================
# QUERY CLAUDE
# ============================================================

def query_claude(image_path):

    encoded_image, media_type = encode_image(
        image_path
    )

    response = client.messages.create(

        model=MODEL,
        max_tokens=200,
        messages=[
            {
                "role": "user",

                "content": [

                    {
                        "type": "image",

                        "source": {
                            "type": "base64",
                            "media_type": media_type,
                            "data": encoded_image
                        }
                    },

                    {
                        "type": "text",
                        "text": PROMPT
                    }
                ]
            }
        ]
    )

    # Usually first content block contains text
    text = ""

    for block in response.content:

        if getattr(block, "type", None) == "text":
            text += block.text

    labels = parse_top5(text)

    return labels, text


# ============================================================
# LOAD CSV / RESUME
# ============================================================

if os.path.exists(OUTPUT_CSV):

    print("Existing result found.")
    print("Resuming:", OUTPUT_CSV)

    df = pd.read_csv(OUTPUT_CSV)

else:

    df = pd.read_csv(INPUT_CSV)


label_columns = [
    "Label 1",
    "Label 2",
    "Label 3",
    "Label 4",
    "Label 5"
]


for col in label_columns:

    if col not in df.columns:
        df[col] = ""


if "status" not in df.columns:
    df["status"] = ""

if "raw_response" not in df.columns:
    df["raw_response"] = ""


print("=" * 60)
print("CLAUDE IMAGE LABELING")
print("=" * 60)

print("Model:", MODEL)
print("Images:", len(df))
print("Folder:", IMAGE_DIR)
print("Output:", OUTPUT_CSV)


# ============================================================
# PROCESS
# ============================================================

for idx in tqdm(
    range(len(df)),
    desc="Claude labeling"
):

    image_name = str(
        df.loc[idx, "image_name"]
    )

    image_path = os.path.join(
        IMAGE_DIR,
        image_name
    )


    # --------------------------------------------------------
    # SKIP COMPLETED
    # --------------------------------------------------------

    already_done = all(

        pd.notna(df.loc[idx, col])

        and

        str(df.loc[idx, col]).strip() != ""

        for col in label_columns
    )

    if already_done:
        continue


    # --------------------------------------------------------
    # FILE CHECK
    # --------------------------------------------------------

    if not os.path.exists(image_path):

        print(
            "\nImage not found:",
            image_path
        )

        df.loc[
            idx,
            "status"
        ] = "image_not_found"

        df.to_csv(
            OUTPUT_CSV,
            index=False
        )

        continue


    # --------------------------------------------------------
    # RETRIES
    # --------------------------------------------------------

    success = False

    for attempt in range(MAX_RETRIES):

        try:

            labels, raw_response = query_claude(
                image_path
            )

            df.loc[
                idx,
                "raw_response"
            ] = raw_response


            if labels is None or len(labels) != 5:

                raise ValueError(
                    f"Could not parse labels: {raw_response}"
                )


            # -----------------------------------------------
            # SAVE LABELS
            # -----------------------------------------------

            for j in range(5):

                df.loc[
                    idx,
                    f"Label {j+1}"
                ] = labels[j]


            df.loc[
                idx,
                "status"
            ] = "success"

            success = True

            break


        except Exception as e:

            print(
                f"\nError on {image_name}"
            )

            print(
                f"Attempt {attempt+1}/{MAX_RETRIES}"
            )

            print(e)

            df.loc[
                idx,
                "status"
            ] = f"failed_attempt_{attempt+1}"


            error_text = str(e)

            # temporary errors / rate limiting
            if (
                "429" in error_text
                or "529" in error_text
                or "overloaded" in error_text.lower()
                or "rate" in error_text.lower()
            ):

                delay = min(
                    60,
                    2 ** attempt
                )

                print(
                    f"Retrying in {delay}s..."
                )

                time.sleep(delay)

            else:

                time.sleep(
                    2 ** attempt
                )


    # --------------------------------------------------------
    # SAVE AFTER EACH IMAGE
    # --------------------------------------------------------

    df.to_csv(
        OUTPUT_CSV,
        index=False
    )


    if not success:

        print(
            "FAILED:",
            image_name
        )


    time.sleep(
        SLEEP_BETWEEN_REQUESTS
    )


# ============================================================
# FINAL SAVE
# ============================================================

df.to_csv(
    OUTPUT_CSV,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

success_count = (
    df["status"] == "success"
).sum()

failed_count = (
    df["status"] != "success"
).sum()


print()
print("=" * 60)
print("FINISHED")
print("=" * 60)

print("Total:", len(df))
print("Successful:", success_count)
print("Failed:", failed_count)

print(
    "Saved:",
    OUTPUT_CSV
)